# 🏙️ Next Hot Neighborhood Predictor - Data Collection

## Project Overview

Welcome to the data collection phase of the "Next Hot Neighborhood Predictor" project!

### 🎯 Objective
Identify up-and-coming neighborhoods before they become expensive by analyzing:
1. **Alternative Data Sources** (Building permits, lifestyle amenities)
2. **Safety Trends** (Crime data)
3. **Geospatial Patterns** (Location-based clustering)

### 📊 Data Sources We'll Collect
- **Yelp API**: Coffee shops, breweries, restaurants (the "Hipster Index")
- **City Open Data**: Building permits, business licenses
- **Police Data**: Crime incidents and trends
- **Census Data** (optional): Demographics and income

### 🚀 What You'll Learn
- API authentication and data collection
- Web scraping from government portals
- Data validation and cleaning
- Geospatial data handling

---

**Author**: Sai Abhinav Pattoor  
**Date**: December 11, 2025

## 📦 Step 1: Import Libraries and Setup

First, we'll import all the necessary libraries. If you get import errors, run the cell below to install missing packages.

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from pathlib import Path

# API and Web Scraping
import requests
from dotenv import load_dotenv

# Data Visualization (for quick checks)
import matplotlib.pyplot as plt
import seaborn as sns

# Geospatial
try:
    import geopandas as gpd
    print("✓ GeoPandas loaded")
except ImportError:
    print("⚠️  GeoPandas not installed. Run: pip install geopandas")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")
print(f"📅 Notebook run date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🌆 Step 2: Select Your Target City

Choose the city you want to analyze. I recommend starting with cities that have excellent open data portals.

In [ ]:
# City Data Portal Configuration
CITY_CONFIG = {
    "austin": {
        "name": "Austin, TX",
        "coordinates": (30.2672, -97.7431),
        "data_portal": "https://data.austintexas.gov",
        "crime_data": "https://data.austintexas.gov/Public-Safety/Crime-Reports/fdj4-gpfu",
        "permits_data": "https://data.austintexas.gov/Building-and-Development/Issued-Construction-Permits/3syk-w9eu",
        "population": 978908,
        "notes": "Excellent data quality, JSON API available"
    },
    "chicago": {
        "name": "Chicago, IL",
        "coordinates": (41.8781, -87.6298),
        "data_portal": "https://data.cityofchicago.org",
        "crime_data": "https://data.cityofchicago.org/Public-Safety/Crimes-2020-to-Present/5cd6-ry5g",
        "permits_data": "https://data.cityofchicago.org/Buildings/Building-Permits/ydr8-5enu",
        "population": 2746388,
        "notes": "Large dataset, very comprehensive"
    },
    "seattle": {
        "name": "Seattle, WA",
        "coordinates": (47.6062, -122.3321),
        "data_portal": "https://data.seattle.gov",
        "crime_data": "https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5",
        "permits_data": "https://data.seattle.gov/Permitting/Building-Permits-Current/mags-97de",
        "population": 749256,
        "notes": "Clean data, well-documented APIs"
    },
    "atlanta": {
        "name": "Atlanta, GA",
        "coordinates": (33.7490, -84.3880),
        "data_portal": "http://opendata.atlantapd.org",
        "crime_data": "http://opendata.atlantapd.org/Crimedata/Default.aspx",
        "permits_data": "https://atlgis.atlantaga.gov",
        "population": 498715,
        "notes": "Good for southern US analysis"
    }
}

# 👉 SELECT YOUR CITY HERE
SELECTED_CITY = "austin"  # Change to: "austin", "chicago", "seattle", or "atlanta"

# Load city configuration
city_config = CITY_CONFIG[SELECTED_CITY]
print(f"🎯 Selected City: {city_config['name']}")
print(f"📍 Coordinates: {city_config['coordinates']}")
print(f"👥 Population: {city_config['population']:,}")
print(f"🌐 Data Portal: {city_config['data_portal']}")
print(f"\n💡 {city_config['notes']}")

## 🔑 Step 3: Configure API Keys

Before collecting data, we need to set up API keys. Follow these instructions:

### Getting Your Yelp API Key (Required)
1. Go to [Yelp Fusion](https://www.yelp.com/developers)
2. Click "Create App"
3. Fill in the form (use any website URL, doesn't have to be real)
4. Copy your **API Key**
5. Create a file named `.env` in the project root
6. Add: `YELP_API_KEY=your_key_here`

### Optional APIs
- **Google Places**: [Get key here](https://developers.google.com/maps/documentation/places/web-service)
- **US Census**: [Get key here](https://api.census.gov/data/key_signup.html)

Run the cell below to verify your API keys are loaded.

In [ ]:
# Verify API Keys
def check_api_keys():
    """Check which API keys are configured"""
    keys = {
        'Yelp': os.getenv('YELP_API_KEY'),
        'Google Places': os.getenv('GOOGLE_PLACES_API_KEY'),
        'Census': os.getenv('CENSUS_API_KEY')
    }
    
    print("API Key Status:")
    print("-" * 50)
    for service, key in keys.items():
        if key and key != 'your_' + service.lower().replace(' ', '_') + '_api_key_here':
            print(f"✅ {service}: Configured")
        else:
            status = "⚠️  Not configured" if service == "Yelp" else "⚪ Optional (not configured)"
            print(f"{status} {service}")
    print("-" * 50)
    
    # Return Yelp key for use
    return keys['Yelp']

YELP_API_KEY = check_api_keys()

if not YELP_API_KEY:
    print("\n❌ CRITICAL: Yelp API key is required!")
    print("📝 Create a .env file with: YELP_API_KEY=your_actual_key")
else:
    print(f"\n✨ Ready to collect data for {city_config['name']}!")

## ☕ Step 4: Collect Yelp Data (The "Hipster Index")

Now we'll use the Yelp Fusion API to collect data about lifestyle amenities. We're looking for:
- **Specialty Coffee Shops** (Blue Bottle, Stumptown, etc.)
- **Craft Breweries** 
- **Organic/Vegan Restaurants**
- **Yoga Studios**

These are leading indicators of neighborhood gentrification!

In [ ]:
from typing import List, Dict
import time

def search_yelp_businesses(api_key: str, term: str, location: str, limit: int = 50) -> List[Dict]:
    """
    Search for businesses on Yelp
    
    Args:
        api_key: Your Yelp API key
        term: Search term (e.g., "coffee", "brewery")
        location: City name (e.g., "Austin, TX")
        limit: Number of results (max 50 per request)
    
    Returns:
        List of business dictionaries
    """
    url = "https://api.yelp.com/v3/businesses/search"
    headers = {"Authorization": f"Bearer {api_key}"}
    
    all_businesses = []
    offset = 0
    max_offset = 1000  # Yelp API limit
    
    while offset < max_offset:
        params = {
            "term": term,
            "location": location,
            "limit": limit,
            "offset": offset
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            response.raise_for_status()
            data = response.json()
            
            businesses = data.get('businesses', [])
            if not businesses:
                break
                
            all_businesses.extend(businesses)
            print(f"  📥 Fetched {len(businesses)} businesses (Total: {len(all_businesses)})")
            
            # Check if we've reached the end
            if len(businesses) < limit:
                break
                
            offset += limit
            time.sleep(0.2)  # Rate limiting: 5 requests/second max
            
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Error: {e}")
            break
    
    return all_businesses

def extract_business_info(businesses: List[Dict]) -> pd.DataFrame:
    """Convert Yelp API response to a clean DataFrame"""
    data = []
    
    for biz in businesses:
        data.append({
            'id': biz.get('id'),
            'name': biz.get('name'),
            'rating': biz.get('rating'),
            'review_count': biz.get('review_count'),
            'price': biz.get('price', 'N/A'),
            'latitude': biz.get('coordinates', {}).get('latitude'),
            'longitude': biz.get('coordinates', {}).get('longitude'),
            'address': biz.get('location', {}).get('address1'),
            'city': biz.get('location', {}).get('city'),
            'zip_code': biz.get('location', {}).get('zip_code'),
            'categories': ', '.join([cat['title'] for cat in biz.get('categories', [])]),
            'url': biz.get('url')
        })
    
    return pd.DataFrame(data)

print("✅ Yelp API functions defined!")

In [ ]:
# Categories to search for (the "gentrification indicators")
SEARCH_CATEGORIES = {
    'coffee': 'Coffee & Tea',
    'breweries': 'Breweries',
    'vegan': 'Vegan Restaurants',
    'yoga': 'Yoga',
    'organic': 'Organic Stores'
}

# Collect data for each category
all_yelp_data = {}

if YELP_API_KEY:
    print(f"🔍 Searching Yelp in {city_config['name']}...\n")
    
    for term, description in SEARCH_CATEGORIES.items():
        print(f"📍 Searching for: {description}")
        businesses = search_yelp_businesses(
            api_key=YELP_API_KEY,
            term=term,
            location=city_config['name'],
            limit=50
        )
        
        df = extract_business_info(businesses)
        all_yelp_data[term] = df
        print(f"  ✅ Found {len(df)} locations\n")
    
    # Combine all categories
    yelp_combined = pd.concat([
        df.assign(category=cat) for cat, df in all_yelp_data.items()
    ], ignore_index=True)
    
    print(f"📊 Total businesses collected: {len(yelp_combined)}")
    print(f"📍 Unique locations: {yelp_combined['id'].nunique()}")
    
else:
    print("❌ Skipping Yelp collection - API key not configured")
    yelp_combined = pd.DataFrame()

In [ ]:
# Preview the Yelp data
if not yelp_combined.empty:
    print("🔍 Sample of collected Yelp data:\n")
    display(yelp_combined.head(10))
    
    print("\n📊 Category breakdown:")
    print(yelp_combined['category'].value_counts())
    
    print("\n⭐ Rating statistics:")
    print(yelp_combined['rating'].describe())

## 💾 Step 5: Save Collected Data

Let's save our collected data to CSV files for the next phase of analysis.

In [ ]:
# Create data directory structure
data_dir = Path("../data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

# Save Yelp data
if not yelp_combined.empty:
    timestamp = datetime.now().strftime('%Y%m%d')
    filename = f"yelp_{SELECTED_CITY}_{timestamp}.csv"
    filepath = data_dir / filename
    
    yelp_combined.to_csv(filepath, index=False)
    print(f"✅ Saved Yelp data: {filepath}")
    print(f"   📊 {len(yelp_combined)} rows × {len(yelp_combined.columns)} columns")
    print(f"   💾 File size: {filepath.stat().st_size / 1024:.2f} KB")
else:
    print("⚠️  No Yelp data to save")

# Save city configuration
config_file = data_dir / f"city_config_{SELECTED_CITY}.json"
with open(config_file, 'w') as f:
    json.dump(city_config, f, indent=2)
print(f"\n✅ Saved city configuration: {config_file}")

## 🎯 Next Steps

Great work! You've completed the initial data collection phase. Here's what comes next:

### ✅ What We've Accomplished
1. ✓ Set up the project environment
2. ✓ Configured API access
3. ✓ Collected lifestyle amenity data from Yelp
4. ✓ Saved raw data for analysis

### 🔜 Coming Up in Future Notebooks

**Notebook 02: Data Cleaning & Enrichment**
- Handle missing values and duplicates
- Geocode addresses to latitude/longitude
- Add zip code and neighborhood boundaries
- Collect crime data from city portals
- Collect building permit data

**Notebook 03: Feature Engineering**
- Calculate the "Hipster Index" for each neighborhood
- Aggregate data by zip code / census tract
- Create time-series features
- Engineer predictive features

**Notebook 04: Exploratory Data Analysis**
- Visualize data distributions
- Find correlations
- Identify patterns and outliers

**Notebook 05: Geospatial Visualization**
- Create interactive maps with Folium/Kepler.gl
- Heatmaps of amenity density
- Identify "up-and-coming" neighborhoods

### 💡 Pro Tips
- Run this notebook weekly to collect fresh data
- Save different versions with timestamps
- Document any API changes or issues

---

**🎉 Congratulations on completing Data Collection!**

Ready to move forward? Let me know and we'll tackle data cleaning next!